In [1]:
import numpy as np
import pandas as pd
import optuna
import xgboost as xgb

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

d:\UB Technology\PY-P\myenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
df = pd.read_csv("../Data/cleaned.csv")

X = df.drop("Exited", axis=1)
y = df["Exited"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
def objective(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 800),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),

        # 🔥 GPU ENABLED
        "tree_method": "hist",
        "device": "cuda",

        "random_state": 42,
        "eval_metric": "logloss"
    }

    model = xgb.XGBClassifier(**params)

    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    return accuracy_score(y_test, preds)

In [ ]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)
print("\n🔥 BEST PARAMETERS:")
print(study.best_params)

[I 2026-04-26 12:23:02,563] A new study created in memory with name: no-name-204ad310-0c33-4705-8903-524642f7b229
[I 2026-04-26 12:23:02,987] Trial 0 finished with value: 0.869 and parameters: {'n_estimators': 442, 'max_depth': 5, 'learning_rate': 0.14883459877689817, 'subsample': 0.6006775676247011, 'colsample_bytree': 0.6731476861212186, 'gamma': 3.687503882292263, 'min_child_weight': 4}. Best is trial 0 with value: 0.869.
[I 2026-04-26 12:23:03,561] Trial 1 finished with value: 0.8685 and parameters: {'n_estimators': 648, 'max_depth': 4, 'learning_rate': 0.014311912402042052, 'subsample': 0.6312504460793645, 'colsample_bytree': 0.8784187407049586, 'gamma': 4.431584239492993, 'min_child_weight': 1}. Best is trial 0 with value: 0.869.
[I 2026-04-26 12:23:03,994] Trial 2 finished with value: 0.8625 and parameters: {'n_estimators': 752, 'max_depth': 6, 'learning_rate': 0.1417214478229146, 'subsample': 0.9631028344468171, 'colsample_bytree': 0.8808969252482228, 'gamma': 0.771857540178466


🔥 BEST PARAMETERS:
{'n_estimators': 589, 'max_depth': 8, 'learning_rate': 0.11644907370744374, 'subsample': 0.7144673257107479, 'colsample_bytree': 0.7230570771644306, 'gamma': 4.986021180381018, 'min_child_weight': 2}


In [ ]:
best_params = study.best_params

best_model = xgb.XGBClassifier(
    **best_params,
    tree_method="hist",
    device="cuda",
    random_state=42
)

best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print("\n✅ FINAL ACCURACY:", acc)

print("\n📊 CLASSIFICATION REPORT:")
print(classification_report(y_test, y_pred))


✅ FINAL ACCURACY: 0.8725

📊 CLASSIFICATION REPORT:
              precision    recall  f1-score   support

           0       0.89      0.96      0.92      1593
           1       0.78      0.52      0.62       407

    accuracy                           0.87      2000
   macro avg       0.83      0.74      0.77      2000
weighted avg       0.87      0.87      0.86      2000

